# optimizer-state-tensor-buffers — faded example 1: Allocate per-param zero buffer list with zeros_like

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-state-tensor-buffers`. The last cell reports your progress on the `Optimizer: Per-param state buffers` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Per-param state buffers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-state-tensor-buffers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-state-tensor-buffers"
DD_SUBTOPIC = "Optimizer: Per-param state buffers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A momentum or EMA buffer list is created at optimizer init time using `[t.zeros_like(p) for p in self.params]`. The `zeros_like` call copies shape, dtype, and device from each parameter automatically, and initializes every element to zero.

## Faded exercise 1

The `__init__` has `self.params = list(params)` already. Add the momentum buffer allocation: `self.buf` should be a list of zero-initialized tensors, one per parameter, using `t.zeros_like`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

class MomentumSGD:
    def __init__(self, params, lr, momentum=0.9):
        self.params   = list(params)
        self.lr       = lr
        self.momentum = momentum
        self.buf = [t.zeros_like(p) for p in self.params]

    @t.inference_mode()
    def step(self):
        for p, b in zip(self.params, self.buf):
            if p.grad is None: continue
            b.copy_(self.momentum * b + p.grad)
            p -= self.lr * b

    def zero_grad(self):
        for p in self.params: p.grad = None

# --- run it ---
t.manual_seed(0)
model = nn.Linear(3, 2)
opt = MomentumSGD(model.parameters(), lr=0.01)
print(f'num buffers: {len(opt.buf)}')                   # 2
print(f'buf[0] shape: {opt.buf[0].shape}')              # [2, 3]
print(f'buf[0] all zero: {opt.buf[0].abs().max().item() == 0.0}')  # True


def _test():
    import torch as t
    import torch.nn as nn
    t.manual_seed(0)
    model = nn.Linear(3, 2)
    opt = MomentumSGD(model.parameters(), lr=0.01)
    assert isinstance(opt.buf, list), 'buf must be a list'
    assert len(opt.buf) == 2, 'Linear(3,2) has 2 param tensors'
    for p, b in zip(opt.params, opt.buf):
        assert p.shape == b.shape, f'Shape mismatch: {p.shape} vs {b.shape}'
        assert p.dtype == b.dtype, f'dtype mismatch: {p.dtype} vs {b.dtype}'
        assert b.requires_grad is False, 'Buffer should not require grad'
        assert b.abs().max().item() == 0.0, 'Buffer should be all zeros at init'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class MomentumSGD:
    def __init__(self, params, lr, momentum=0.9):
        self.params   = list(params)
        self.lr       = lr
        self.momentum = momentum
        self.buf = [t.zeros_like(p) for p in self.params]

    @t.inference_mode()
    def step(self):
        for p, b in zip(self.params, self.buf):
            if p.grad is None: continue
            b.copy_(self.momentum * b + p.grad)
            p -= self.lr * b

    def zero_grad(self):
        for p in self.params: p.grad = None

# --- run it ---
t.manual_seed(0)
model = nn.Linear(3, 2)
opt = MomentumSGD(model.parameters(), lr=0.01)
print(f'num buffers: {len(opt.buf)}')                   # 2
print(f'buf[0] shape: {opt.buf[0].shape}')              # [2, 3]
print(f'buf[0] all zero: {opt.buf[0].abs().max().item() == 0.0}')  # True
```
</details>